# 🦵 MRNet Full Training Pipeline
**ACL / Meniscus / Abnormality Detection from Knee MRI**

This notebook runs the complete 9-model training pipeline on Google Colab's free GPU.

## Before you start — 3 things to do:
1. **Runtime → Change runtime type → T4 GPU** (free)
2. **Upload your MRNet data to Google Drive** (see Cell 3 for folder structure)
3. **Run all cells top to bottom** — each cell builds on the previous one

---
**Estimated time:** ~2-3 hours for all 9 models (3 conditions × 3 planes) × 30 epochs

In [ ]:
# ── Cell 1: Check we have a GPU ──────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✅ GPU detected: {gpu_name}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌ No GPU! Go to Runtime → Change runtime type → T4 GPU and restart.')
    raise SystemExit('Please enable GPU first.')

In [ ]:
# ── Cell 2: Mount Google Drive ───────────────────────────────────────────────
# This connects Colab to your Google Drive so we can read data and save results.
# You will see a pop-up asking you to sign in — that is normal.
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

## 📁 Data Setup

Before running Cell 3, upload your MRNet data to Google Drive in this structure:

```
My Drive/
└── MRNet/
    └── mrnet/
        ├── train/
        │   ├── axial/      ← .npy files
        │   ├── coronal/
        │   └── sagittal/
        ├── valid/
        │   ├── axial/
        │   ├── coronal/
        │   └── sagittal/
        ├── train-acl.csv
        ├── train-meniscus.csv
        ├── train-abnormal.csv
        ├── valid-acl.csv
        ├── valid-meniscus.csv
        └── valid-abnormal.csv
```

**Tip:** Zip the whole `mrnet/` folder, upload the zip to Drive, then unzip it using Cell 3.

In [ ]:
# ── Cell 3: Set up data path ─────────────────────────────────────────────────
import os

# If you uploaded a zip, unzip it first:
DRIVE_ZIP = '/content/drive/MyDrive/MRNet/mrnet.zip'
if os.path.exists(DRIVE_ZIP):
    print('📦 Unzipping MRNet data (this takes a few minutes)...')
    os.system(f'unzip -q "{DRIVE_ZIP}" -d /content/mrnet_data')
    print('✅ Unzipped!')
else:
    print('No zip found — assuming data is already extracted.')

# Point to the data directory
DATA_DIR = '/content/mrnet_data/mrnet'
if not os.path.isdir(os.path.join(DATA_DIR, 'train')):
    # Try the Drive path directly if not unzipped
    DATA_DIR = '/content/drive/MyDrive/MRNet/mrnet'

print(f'📂 Data directory: {DATA_DIR}')
print(f'   Train folder exists: {os.path.isdir(os.path.join(DATA_DIR, "train"))}')
print(f'   Valid folder exists: {os.path.isdir(os.path.join(DATA_DIR, "valid"))}')

In [ ]:
# ── Cell 4: Clone YOUR private repo (contains all the training code) ──────────
# This is your private mrnet-learning repo — only you have access to it.
import os

REPO_URL = 'https://github.com/Dublindeveloper/mrnet-learning.git'
REPO_DIR = '/content/mrnet-learning'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Switch to the branch with all our implemented code
!cd {REPO_DIR} && git checkout local-learning-end-to-end

print(f'\n✅ Repository ready at {REPO_DIR}')
!ls {REPO_DIR}/code/src/modules/

In [ ]:
# ── Cell 5: Install Python dependencies ──────────────────────────────────────
!pip install -q torch torchvision tqdm scikit-learn matplotlib opencv-python-headless pandas numpy
print('✅ All packages installed')

In [ ]:
# ── Cell 6: Sanity check — load one exam to make sure data is readable ────────
import sys
sys.path.insert(0, f'{REPO_DIR}/code/src')

from modules.data_preprocessing_transformation import MRNetDataset, get_augmentation_transform

test_ds = MRNetDataset(DATA_DIR, 'acl', 'sagittal', 'train')
exam, label = test_ds[0]
print(f'✅ Data check passed!')
print(f'   Dataset size: {len(test_ds)} exams')
print(f'   Exam shape:   {exam.shape}  (slices × 1 × H × W)')
print(f'   pos_weight:   {test_ds.pos_weight.item():.2f}  (penalty for missing a tear)')

## 🎓 Training All 9 Models

We train one model per condition-plane combination:

| | Axial | Coronal | Sagittal |
|---|---|---|---|
| **ACL** | Model 1 | Model 2 | Model 3 |
| **Meniscus** | Model 4 | Model 5 | Model 6 |
| **Abnormal** | Model 7 | Model 8 | Model 9 |

Each model uses **two-phase training**:
- **Phase 1 (10 epochs):** Only the new classification head learns. ResNet18 backbone is frozen.
- **Phase 2 (20 epochs):** The whole model fine-tunes together at a slower learning rate.

Checkpoints are saved to Google Drive after every best epoch.

In [ ]:
# ── Cell 7: Configure training ───────────────────────────────────────────────
import os

# All results will be saved here in your Google Drive
OUTPUT_DIR     = '/content/drive/MyDrive/MRNet/results'
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
RESULTS_DIR    = os.path.join(OUTPUT_DIR, 'evaluation')
VIS_DIR        = os.path.join(OUTPUT_DIR, 'visualisations')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)
os.makedirs(VIS_DIR,        exist_ok=True)

CONDITIONS = ['acl', 'meniscus', 'abnormal']
PLANES     = ['axial', 'coronal', 'sagittal']

# Training hyperparameters
EPOCHS_PHASE1 = 10
EPOCHS_PHASE2 = 20
LR_PHASE1     = 1e-3
LR_PHASE2     = 1e-4

print(f'📁 Checkpoints → {CHECKPOINT_DIR}')
print(f'📊 Conditions:  {CONDITIONS}')
print(f'🏔️  Planes:      {PLANES}')
print(f'⏱️  Total models: {len(CONDITIONS) * len(PLANES)}')
print(f'   Epochs/model: {EPOCHS_PHASE1 + EPOCHS_PHASE2}')

In [ ]:
# ── Cell 8: Train all 9 models ───────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm  # notebook-friendly progress bar

from modules.data_preprocessing_transformation import MRNetDataset, get_augmentation_transform, stack_channels
from modules.baseline_models import create_baseline_model

device = torch.device('cuda')  # We confirmed GPU in Cell 1
print(f'Training on: {torch.cuda.get_device_name(0)}\n')

def train_one_model(condition, plane):
    """Train a single condition-plane model through both phases."""
    model_name   = f'{condition}_{plane}_baseline'
    ckpt_dir     = os.path.join(CHECKPOINT_DIR, model_name)
    os.makedirs(ckpt_dir, exist_ok=True)

    print(f'\n' + '='*60)
    print(f'  Training: {model_name.upper()}')
    print('='*60)

    # Datasets
    train_ds = MRNetDataset(DATA_DIR, condition, plane, 'train',
                            transform=get_augmentation_transform(True))
    val_ds   = MRNetDataset(DATA_DIR, condition, plane, 'valid')
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=2)

    model     = create_baseline_model().to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=train_ds.pos_weight.to(device))

    def run_epoch(loader, optimizer=None, training=True):
        model.train(training)
        total_loss, all_probs, all_labels = 0, [], []
        desc = 'Train' if training else 'Val'
        for exam, label in tqdm(loader, desc=f'    {desc}', leave=False):
            exam_in = stack_channels(exam.squeeze(0)).unsqueeze(0).to(device)
            label   = label.to(device)
            if training:
                optimizer.zero_grad()
            with torch.set_grad_enabled(training):
                out  = model(exam_in)
                loss = criterion(out, label)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            all_probs.append(torch.sigmoid(out).item())
            all_labels.append(label.item())
        avg_loss = total_loss / len(loader)
        try:
            auc = roc_auc_score(all_labels, all_probs)
        except Exception:
            auc = 0.5
        return avg_loss, auc

    def train_phase(phase_name, n_epochs, lr):
        optimizer     = optim.Adam(model.get_trainable_params(), lr=lr)
        best_val_loss = float('inf')
        print(f'\n  Phase {phase_name}: {n_epochs} epochs @ lr={lr}')
        for epoch in range(1, n_epochs + 1):
            tr_loss, tr_auc = run_epoch(train_loader, optimizer, training=True)
            va_loss, va_auc = run_epoch(val_loader,   training=False)
            print(f'  Ep {epoch:02d}/{n_epochs} | '
                  f'Train {tr_loss:.4f}/{tr_auc:.3f} | '
                  f'Val {va_loss:.4f}/{va_auc:.3f}', end='')
            if va_loss < best_val_loss:
                best_val_loss = va_loss
                ckpt_path = os.path.join(ckpt_dir, f'best_phase{phase_name}.pth')
                torch.save({'epoch': epoch,
                            'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(),
                            'loss': va_loss}, ckpt_path)
                print(' 💾 saved')
            else:
                print()
        return best_val_loss

    # Phase 1: frozen backbone
    model.freeze_backbone()
    train_phase('1', EPOCHS_PHASE1, LR_PHASE1)

    # Phase 2: full fine-tuning
    model.unfreeze_backbone()
    train_phase('2', EPOCHS_PHASE2, LR_PHASE2)

    print(f'\n  ✅ Done: {model_name}')


# Train all 9 models
for condition in CONDITIONS:
    for plane in PLANES:
        train_one_model(condition, plane)

print('\n🎉 ALL 9 MODELS TRAINED!')

In [ ]:
# ── Cell 9: Evaluate all 9 models ────────────────────────────────────────────
import sys
sys.path.insert(0, f'{REPO_DIR}/code/src')
from evaluation import evaluate_architecture

print('📊 EVALUATION RESULTS')
print('='*70)

summary = {}
for condition in CONDITIONS:
    result = evaluate_architecture(
        condition=condition,
        architecture='baseline',
        checkpoint_dir=CHECKPOINT_DIR,
        data_dir=DATA_DIR,
        output_dir=RESULTS_DIR
    )
    if result:
        metrics, _ = result
        summary[condition] = metrics

print('\n📋 SUMMARY TABLE')
print(f'{"Condition":<12} {"AUC":>8} {"95% CI":>18} {"Sensitivity":>13}')
print('-' * 55)
for cond, m in summary.items():
    auc_v, auc_lo, auc_hi = m['auc']
    sens_v = m['sensitivity'][0]
    print(f'{cond.upper():<12} {auc_v:>8.4f} ({auc_lo:.3f} – {auc_hi:.3f}) {sens_v:>13.4f}')

In [ ]:
# ── Cell 10: Generate Grad-CAM heatmaps ──────────────────────────────────────
from explainability import run_gradcam_pipeline

for condition in CONDITIONS:
    for plane in PLANES:
        ckpt = os.path.join(CHECKPOINT_DIR, f'{condition}_{plane}_baseline', 'best_phase2.pth')
        if not os.path.exists(ckpt):
            continue
        print(f'\n🔥 Grad-CAM: {condition} {plane}...')
        try:
            run_gradcam_pipeline(
                condition=condition,
                plane=plane,
                architecture='baseline',
                checkpoint_path=ckpt,
                data_dir=DATA_DIR,
                output_dir=VIS_DIR,
                num_examples=2
            )
        except Exception as e:
            print(f'  ⚠️ Skipped ({e})')

print(f'\n✅ All Grad-CAM visualisations saved to {VIS_DIR}')

In [ ]:
# ── Cell 11: Preview Grad-CAM images inline ───────────────────────────────────
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

imgs = glob.glob(os.path.join(VIS_DIR, 'acl_sagittal', 'confident_correct_tear*.png'))
if imgs:
    img = mpimg.imread(imgs[0])
    plt.figure(figsize=(16, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('ACL Sagittal — Grad-CAM (Confident Correct Tear)', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('No Grad-CAM images found yet — run Cell 10 first.')

## ✅ Done!

All results are saved to your **Google Drive** at `My Drive/MRNet/results/`:

```
results/
├── checkpoints/        ← Trained model weights (.pth files)
├── evaluation/         ← ROC curves and metric summaries
└── visualisations/     ← Grad-CAM heatmap images
```

### Next steps:
1. Download the `checkpoints/` folder from Drive back to your Mac
2. Run `evaluation.py` and `explainability.py` locally with the new checkpoints
3. Assess the Grad-CAM heatmaps clinically — does the red glow sit over the intercondylar notch?
4. Write your clinical validation commentary for the team report 🩺